# Python deconvolution of all genes in the Xin Guo 2011 dataset, checking timing and concordance with Matlab and published dataset

In this notebook, we will test the deconvolve all of the genes in the Xin Guo 2011 dataset.

The goal is to verify that the deconvolution has been successfully ported from the matlab code base.

We will check the timing and compare the peak to trough values to the published paper dataset. 

In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from src.model import Model
from src.config import load_xg_gene_expression_config

In [3]:
# Run the model once to get the dimensions of the final output
# TODO: we shouldn't need to do this once the model is read in, but we
# update this later.
og_config = load_xg_gene_expression_config()
og_model = Model(og_config, 'EDC1', 0.00429)
og_model.deconvolve()

In [4]:
# Load the gene list
genelist = pd.read_csv('allgenes/commongenelist.txt', sep='\t', header=None).values
genelist = genelist.reshape(-1)
genelist

array(['SGV1', 'RTC1', 'SXM1', ..., 'HSP10', 'YLL053C', 'RAD34'],
      dtype=object)

In [5]:
# Load the precomputed gamma values from Xin Guo's deconv.v2 codebase
deconvv2_gene_gammas = pd.read_csv('allgenes/from_deconvv2/allgenes.gm', sep='\t', header=None)
deconvv2_gene_gammas.columns = ['gene', 'orf_name', 'gamma']
deconvv2_gene_gammas = deconvv2_gene_gammas.set_index('gene')

In [8]:
from src.timer import Timer

# Initialize configuration and model
og_config = load_xg_gene_expression_config()
timer = Timer()

# We will store all of the deconvolved f values into this matrix
gene_fs = np.zeros((len(genelist), og_model.f.value.shape[0]))


In [9]:

print(f"Deconvolving {len(genelist)} genes...")
# Loop through all genes and store the deconvolved f values.
for index in range(len(genelist)):

    gene = genelist[index]
    
    # Load the appopriate gamma value
    try:
        gamma = deconvv2_gene_gammas.loc[gene]['gamma']
    except KeyError:
        print(f"No gamma for gene: {gene}")
        continue

    try:
        og_model = Model(og_config, gene, gamma)
    except AttributeError:
        print(f"{gene} was not found. Skipping.")
        continue

    og_model.deconvolve()
    
    gene_fs[index] = og_model.f.value

    if index % 100 == 0:
        print(f"   {index+1}/{len(genelist)} - {timer.get_time()}")



Deconvolving 5670 genes...
   1/5670 - 00:00:02.12
   101/5670 - 00:02:11.65
   201/5670 - 00:03:05.02
   301/5670 - 00:03:57.41
   401/5670 - 00:04:49.96
   501/5670 - 00:05:42.30
   601/5670 - 00:06:33.53
   701/5670 - 00:07:25.40
   801/5670 - 00:08:17.40
   901/5670 - 00:09:06.33
   1001/5670 - 00:09:55.26
   1101/5670 - 00:10:44.10
   1201/5670 - 00:11:32.46
   1301/5670 - 00:12:20.96
   1401/5670 - 00:13:09.30
   1501/5670 - 00:13:57.70
   1601/5670 - 00:14:46.21
   1701/5670 - 00:20:35.64
   1801/5670 - 00:21:24.01
   1901/5670 - 00:22:12.07
   2001/5670 - 00:23:00.79
   2101/5670 - 00:23:49.70
   2201/5670 - 00:24:38.74
   2301/5670 - 00:25:27.64
   2401/5670 - 00:26:16.33
   2501/5670 - 00:27:05.46
   2601/5670 - 00:27:53.90
   2701/5670 - 00:28:42.74
   2801/5670 - 00:29:31.55
   2901/5670 - 00:30:20.54
   3001/5670 - 00:32:57.93
   3101/5670 - 00:33:45.87
   3201/5670 - 00:34:34.16
   3301/5670 - 00:35:23.13
   3401/5670 - 00:36:12.06
   3501/5670 - 00:37:01.15
   3601/5670 

In [14]:
gene_fs_df = pd.DataFrame(gene_fs)
gene_fs_df.to_csv('output/xin-deconvolved_fs.csv', index=False)